# LSTM Multi-Ciudad — Carsharing Demand Estimation

**Estrategia:**  
- **Preentrenamiento:** 9 ciudades (2015) → el modelo aprende patrones generales de demanda  
- **Zero-shot test:** München (2016) → evalúa transferibilidad sin reentrenamiento  
- **Baseline de comparación:** XGBoost en Milán → MAE=0.92, RMSE=1.47, R²=0.57

**Unidad de serie temporal:** cada celda H3 de cada ciudad es una serie independiente.  
**Ventana deslizante:** T=24h → predice demanda en hora t+1.

## 0. Instalación

In [ ]:
# !pip install torch numpy pandas scikit-learn matplotlib pyarrow

## 1. Imports y configuración

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ─── Reproducibilidad ───────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# ─── Configuración principal ─────────────────────────────────────────────────
DATA_PATH = Path('dataset_h3_multicidad.parquet')

# Ciudad de test (transfer learning / zero-shot)
CITY_TEST = 'muenchen'

# Filtrado de celdas: mínimo % de horas con demanda > 0
# (descarta celdas casi siempre vacías)
MIN_ACTIVE_PCT = 0.15   # 15% de horas con al menos 1 viaje

# Ventana temporal para el LSTM
WINDOW_SIZE = 24        # 24h de contexto → predice h+1

# Validación: últimos N días de cada ciudad de entrenamiento
VAL_DAYS = 7

# Hiperparámetros del modelo
HIDDEN_SIZE  = 64
NUM_LAYERS   = 2
DROPOUT      = 0.2
BATCH_SIZE   = 512
LEARNING_RATE = 1e-3
EPOCHS       = 50
PATIENCE     = 8        # early stopping

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Ciudad de test (transfer): {CITY_TEST}')

## 2. Carga del dataset

In [ ]:
df = pd.read_parquet(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])

# Ordenar cronológicamente dentro de cada serie
df.sort_values(['city', 'h3_cell', 'date', 'hour'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Filas totales: {len(df):,}')
print(f'Ciudades: {sorted(df["city"].unique())}')
print(f'Columnas: {list(df.columns)}')

## 3. Filtrado de celdas activas

Dado que el 70-90% de los slots son ceros, eliminamos celdas con muy baja actividad  
para que el modelo no se limite a predecir siempre cero.

In [ ]:
# Porcentaje de horas activas por celda
cell_activity = (
    df.groupby(['city', 'h3_cell'])['target_demanda']
    .apply(lambda x: (x > 0).mean())
    .reset_index(name='pct_active')
)

active_cells = cell_activity[cell_activity['pct_active'] >= MIN_ACTIVE_PCT]

print(f'Celdas antes del filtro: {len(cell_activity):,}')
print(f'Celdas activas (>={MIN_ACTIVE_PCT*100:.0f}%): {len(active_cells):,}')
print(f'\nCeldas activas por ciudad:')
print(active_cells.groupby('city')['h3_cell'].count().to_string())

# Aplicar filtro
df = df.merge(active_cells[['city', 'h3_cell']], on=['city', 'h3_cell'], how='inner')
print(f'\nFilas tras filtro: {len(df):,}')

## 4. Features y normalización

Normalización **por celda** (z-score) para que el modelo aprenda patrones relativos  
y no se confunda con diferencias absolutas de escala entre ciudades.

> ⚠️ Los parámetros de normalización de München se calculan **solo con sus propios datos**  
> (sin filtrar por split) para simular un escenario realista de zero-shot.

In [ ]:
# Features que entran al LSTM en cada paso temporal
# El LSTM recibe la secuencia de (demanda_norm, features_temporales)
FEATURE_COLS = [
    'target_demanda',   # la serie en sí (se normaliza)
    'hour_sin', 'hour_cos',
    'dow_sin', 'dow_cos',
    'is_weekend'
]
N_FEATURES = len(FEATURE_COLS)

# Normalización z-score por celda (solo sobre target_demanda)
# Las features temporales (sin/cos) ya están en [-1, 1]
cell_stats = (
    df.groupby(['city', 'h3_cell'])['target_demanda']
    .agg(['mean', 'std'])
    .reset_index()
)
cell_stats['std'] = cell_stats['std'].replace(0, 1)  # evitar división por cero

df = df.merge(cell_stats, on=['city', 'h3_cell'], how='left')
df['demand_norm'] = (df['target_demanda'] - df['mean']) / df['std']

# Actualizar la columna de demanda en FEATURE_COLS con el valor normalizado
df['target_demanda_orig'] = df['target_demanda'].copy()
df['target_demanda']      = df['demand_norm']

print('Normalización completada.')
print(f'Estadísticas de demand_norm:')
print(df['demand_norm'].describe())

## 5. Construcción de ventanas deslizantes (PyTorch Dataset)

In [ ]:
class SlidingWindowDataset(Dataset):
    """
    Para cada serie (city, h3_cell) crea ventanas deslizantes de tamaño T.
    X: [T, n_features]  →  y: demanda normalizada en t+1 (escalar)
    También guarda mean/std para desnormalizar predicciones.
    """
    def __init__(self, df_subset: pd.DataFrame, T: int = 24):
        self.T = T
        windows, targets, denorm_params = [], [], []

        groups = df_subset.groupby(['city', 'h3_cell'])
        for (city, cell), grp in groups:
            grp = grp.sort_values(['date', 'hour'])
            vals = grp[FEATURE_COLS].values.astype(np.float32)  # (n_steps, n_features)
            mu   = grp['mean'].iloc[0]
            sigma = grp['std'].iloc[0]

            # Crear ventanas
            for i in range(T, len(vals)):
                windows.append(vals[i-T:i])         # [T, n_features]
                targets.append(vals[i, 0])           # demanda normalizada en t
                denorm_params.append((mu, sigma))

        self.X = torch.tensor(np.array(windows), dtype=torch.float32)
        self.y = torch.tensor(np.array(targets),  dtype=torch.float32)
        self.denorm = denorm_params   # lista de (mean, std) por muestra

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def denormalize(y_norm: np.ndarray, denorm_params: list) -> np.ndarray:
    """Convierte predicciones normalizadas a escala original de viajes."""
    result = np.zeros_like(y_norm)
    for i, (mu, sigma) in enumerate(denorm_params):
        result[i] = y_norm[i] * sigma + mu
    return np.clip(result, 0, None)  # demanda no puede ser negativa


print('Clase SlidingWindowDataset definida.')

## 6. Split Train / Validación / Test

- **Train:** 9 ciudades (todo excepto München), sin los últimos `VAL_DAYS` días  
- **Val:** 9 ciudades, últimos `VAL_DAYS` días (para early stopping)  
- **Test zero-shot:** München completo (nunca visto durante el entrenamiento)

In [ ]:
df_pretrain = df[df['city'] != CITY_TEST].copy()
df_zeroshot = df[df['city'] == CITY_TEST].copy()

# Split temporal dentro del conjunto de preentrenamiento
cutoff_dates = (
    df_pretrain.groupby('city')['date']
    .max()
    .apply(lambda d: d - pd.Timedelta(days=VAL_DAYS))
)

train_mask = df_pretrain.apply(
    lambda row: row['date'] <= cutoff_dates[row['city']], axis=1
)

df_train = df_pretrain[train_mask].copy()
df_val   = df_pretrain[~train_mask].copy()

print(f'Train:     {len(df_train):,} filas  ({df_train["city"].nunique()} ciudades)')
print(f'Val:       {len(df_val):,} filas')
print(f'Test:      {len(df_zeroshot):,} filas  ({CITY_TEST})')

In [ ]:
# Construir datasets (puede tardar ~1-2 min)
print('Construyendo ventanas de entrenamiento...')
train_ds = SlidingWindowDataset(df_train, T=WINDOW_SIZE)
print(f'  Train windows: {len(train_ds):,}')

print('Construyendo ventanas de validación...')
val_ds = SlidingWindowDataset(df_val, T=WINDOW_SIZE)
print(f'  Val windows:   {len(val_ds):,}')

print('Construyendo ventanas de test (München)...')
test_ds = SlidingWindowDataset(df_zeroshot, T=WINDOW_SIZE)
print(f'  Test windows:  {len(test_ds):,}')

# DataLoaders
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

## 7. Modelo LSTM

Arquitectura con **dos capas LSTM** + cabeza densa.  
En transfer learning, se congelarán las capas LSTM y solo se reentrenará `head`.

In [ ]:
class DemandLSTM(nn.Module):
    """
    LSTM para predicción de demanda de carsharing.

    Separamos explícitamente encoder (LSTM) y head (Dense)
    para facilitar el fine-tuning por transfer learning.
    """
    def __init__(self, n_features: int, hidden_size: int, num_layers: int, dropout: float):
        super().__init__()
        self.encoder = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        # x: [batch, T, n_features]
        out, _ = self.encoder(x)   # out: [batch, T, hidden]
        last    = out[:, -1, :]    # último estado: [batch, hidden]
        return self.head(last).squeeze(-1)  # [batch]


model = DemandLSTM(
    n_features=N_FEATURES,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'\nParámetros totales: {total_params:,}')

## 8. Entrenamiento

In [ ]:
criterion  = nn.HuberLoss(delta=1.0)   # robusto a outliers, mejor que MSE con tantos ceros
optimizer  = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=3, factor=0.5, verbose=True
)

train_losses, val_losses = [], []
best_val_loss = float('inf')
patience_counter = 0
best_state = None

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    model.train()
    epoch_train_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        pred = model(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # evitar exploding gradients
        optimizer.step()
        epoch_train_loss += loss.item() * len(y_batch)
    epoch_train_loss /= len(train_ds)

    # ── Validación ──
    model.eval()
    epoch_val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            pred = model(X_batch)
            epoch_val_loss += criterion(pred, y_batch).item() * len(y_batch)
    epoch_val_loss /= len(val_ds)

    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)
    scheduler.step(epoch_val_loss)

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS} | Train: {epoch_train_loss:.4f} | Val: {epoch_val_loss:.4f}')

    # Early stopping
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_state    = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stopping en epoch {epoch}.')
            break

# Restaurar mejor modelo
model.load_state_dict(best_state)
torch.save(best_state, 'lstm_pretrained.pt')
print(f'\nMejor val loss: {best_val_loss:.4f} — modelo guardado en lstm_pretrained.pt')

In [ ]:
# Curva de aprendizaje
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='Train')
plt.plot(val_losses,   label='Val')
plt.xlabel('Epoch')
plt.ylabel('Huber Loss (escala normalizada)')
plt.title('Curva de aprendizaje — LSTM preentrenado')
plt.legend()
plt.tight_layout()
plt.savefig('curva_aprendizaje.png', dpi=120)
plt.show()

## 9. Evaluación — Función de métricas

In [ ]:
def evaluate(loader: DataLoader, dataset: SlidingWindowDataset, label: str) -> dict:
    """Evalúa el modelo en escala original (viajes reales, no normalizada)."""
    model.eval()
    preds_norm, targets_norm = [], []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(DEVICE)
            preds_norm.append(model(X_batch).cpu().numpy())
            targets_norm.append(y_batch.numpy())

    preds_norm   = np.concatenate(preds_norm)
    targets_norm = np.concatenate(targets_norm)

    # Desnormalizar a escala de viajes reales
    preds_real   = denormalize(preds_norm,   dataset.denorm)
    targets_real = denormalize(targets_norm, dataset.denorm)

    mae  = mean_absolute_error(targets_real, preds_real)
    rmse = np.sqrt(mean_squared_error(targets_real, preds_real))
    r2   = r2_score(targets_real, preds_real)

    print(f'[{label}]  MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}')
    return {'label': label, 'MAE': mae, 'RMSE': rmse, 'R2': r2,
            'preds': preds_real, 'targets': targets_real}

## 10. Resultados en ciudades de preentrenamiento y zero-shot en München

In [ ]:
results_val  = evaluate(val_loader,  val_ds,  'Val (9 ciudades, última semana)')
results_test = evaluate(test_loader, test_ds, 'Zero-shot München')

In [ ]:
# Evaluación por ciudad individual (en val)
print('\n── Métricas por ciudad (split de validación) ──')
city_metrics = []

for city in sorted(df_val['city'].unique()):
    df_city = df_val[df_val['city'] == city]
    ds_city = SlidingWindowDataset(df_city, T=WINDOW_SIZE)
    if len(ds_city) == 0:
        continue
    loader_city = DataLoader(ds_city, batch_size=BATCH_SIZE, shuffle=False)
    m = evaluate(loader_city, ds_city, city)
    city_metrics.append(m)

# Añadir München
city_metrics.append(results_test)

## 11. Tabla comparativa vs XGBoost baseline

In [ ]:
# XGBoost baseline (Milán, del notebook eda_preliminar)
baseline = {'label': 'XGBoost (Milán)', 'MAE': 0.9247, 'RMSE': 1.4663, 'R2': 0.5660}

rows = [baseline] + [
    {'label': f'LSTM {m["label"]}', 'MAE': m['MAE'], 'RMSE': m['RMSE'], 'R2': m['R2']}
    for m in city_metrics
]

df_results = pd.DataFrame(rows).set_index('label').round(4)
print('\n══ TABLA COMPARATIVA ══')
print(df_results.to_string())

## 12. Visualización de predicciones vs real (Milán y München)

In [ ]:
def plot_predictions(targets, preds, city: str, n_hours: int = 168):
    """Muestra las primeras n_hours del split de evaluación."""
    fig, axes = plt.subplots(2, 1, figsize=(14, 6))

    axes[0].plot(targets[:n_hours], label='Real', alpha=0.8)
    axes[0].plot(preds[:n_hours],   label='Predicción LSTM', alpha=0.8)
    axes[0].set_title(f'{city.capitalize()} — Primeras {n_hours}h de evaluación')
    axes[0].set_ylabel('Viajes')
    axes[0].legend()

    axes[1].scatter(targets, preds, alpha=0.2, s=5)
    lim = max(targets.max(), preds.max())
    axes[1].plot([0, lim], [0, lim], 'r--', label='Predicción perfecta')
    axes[1].set_xlabel('Real')
    axes[1].set_ylabel('Predicción')
    axes[1].set_title('Scatter real vs predicción')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(f'predicciones_{city}.png', dpi=120)
    plt.show()


# Buscar métricas de Milán en city_metrics
milano_m = next((m for m in city_metrics if 'milano' in m['label']), None)
if milano_m:
    plot_predictions(milano_m['targets'], milano_m['preds'], 'milano')

plot_predictions(results_test['targets'], results_test['preds'], 'muenchen')

## 13. Fine-tuning en München (opcional)

Congela el encoder (capas LSTM) y reentrenar solo la cabeza densa  
con una fracción pequeña de los datos de München.

In [ ]:
# Fracción de datos de München usada para fine-tuning
FINETUNE_DAYS = 7   # primera semana de München → ajustar cabeza

muen_dates     = sorted(df_zeroshot['date'].unique())
finetune_dates = muen_dates[:FINETUNE_DAYS]

df_muen_ft   = df_zeroshot[df_zeroshot['date'].isin(finetune_dates)]
df_muen_eval = df_zeroshot[~df_zeroshot['date'].isin(finetune_dates)]

print(f'Fine-tuning: {len(df_muen_ft):,} filas ({FINETUNE_DAYS} días)')
print(f'Evaluación:  {len(df_muen_eval):,} filas')

ft_ds     = SlidingWindowDataset(df_muen_ft,   T=WINDOW_SIZE)
eval_ds   = SlidingWindowDataset(df_muen_eval, T=WINDOW_SIZE)
ft_loader = DataLoader(ft_ds,   batch_size=128, shuffle=True)
ev_loader = DataLoader(eval_ds, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# Cargar el mejor modelo preentrenado
model_ft = DemandLSTM(N_FEATURES, HIDDEN_SIZE, NUM_LAYERS, DROPOUT).to(DEVICE)
model_ft.load_state_dict(torch.load('lstm_pretrained.pt', map_location=DEVICE))

# ── Congelar encoder (capas LSTM) ──
for param in model_ft.encoder.parameters():
    param.requires_grad = False

# Solo se optimiza la cabeza
ft_optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_ft.parameters()),
    lr=1e-4
)

print('Parámetros entrenables (solo head):')
for name, p in model_ft.named_parameters():
    if p.requires_grad:
        print(f'  {name}: {p.numel():,}')

In [ ]:
# Fine-tuning: pocas épocas con lr bajo
FT_EPOCHS = 20

for epoch in range(1, FT_EPOCHS + 1):
    model_ft.train()
    ft_loss = 0.0
    for X_batch, y_batch in ft_loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        ft_optimizer.zero_grad()
        pred = model_ft(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        ft_optimizer.step()
        ft_loss += loss.item() * len(y_batch)
    ft_loss /= len(ft_ds)
    if epoch % 5 == 0:
        print(f'FT Epoch {epoch:2d}/{FT_EPOCHS} | Loss: {ft_loss:.4f}')

torch.save(model_ft.state_dict(), 'lstm_finetuned_muenchen.pt')
print('Modelo fine-tuned guardado.')

In [ ]:
# Evaluar: zero-shot vs fine-tuned (en días de München NO usados para FT)
# Reasignar model temporalmente para usar evaluate()
original_model = model
model = model_ft

results_ft = evaluate(ev_loader, eval_ds, 'Fine-tuned München (1 semana FT)')

model = original_model  # restaurar

# ── Resumen final ──
print('\n══ COMPARATIVA FINAL ══')
rows_final = [
    {'Modelo': 'XGBoost Milán (baseline)',          'MAE': 0.9247, 'RMSE': 1.4663, 'R²': 0.5660},
    {'Modelo': f'LSTM zero-shot München',           'MAE': results_test['MAE'], 'RMSE': results_test['RMSE'], 'R²': results_test['R2']},
    {'Modelo': f'LSTM fine-tuned München (7 días)', 'MAE': results_ft['MAE'],  'RMSE': results_ft['RMSE'],  'R²': results_ft['R2']},
]
print(pd.DataFrame(rows_final).set_index('Modelo').round(4).to_string())

---
## Próximos pasos

1. **Variables socioeconómicas**: añadir como features estáticas concatenadas al estado LSTM (o como embedding por celda)
2. **Resolución H3**: probar resolución 7 si hay demasiadas celdas poco activas
3. **Arquitectura**: probar Transformer / Temporal Fusion Transformer (TFT) como alternativa
4. **Fine-tuning progresivo**: descongelar capas LSTM de una en una y medir impacto
5. **Milano como ciudad de test alternativa**: comparar LSTM multiciudad vs XGBoost entrenado solo en Milán